## Final Workflow: ML based integration pipeline (Sorted Neighbourhood Blocker)

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

## Schema Mapping

In [2]:
from PyDI.io import load_parquet, load_csv

amazon_books = load_parquet(
    DATA_DIR / "amazon.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks.parquet",
  name="metabooks"
)

In [3]:
import pandas as pd
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer, util


# embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

def similarity_score(src, tgt):
    """Compute semantic + fuzzy similarity between column names."""
    emb_src = model.encode(src, convert_to_tensor=True)
    emb_tgt = model.encode(tgt, convert_to_tensor=True)
    semantic_sim = util.cos_sim(emb_src, emb_tgt).item()
    fuzzy_sim = fuzz.token_sort_ratio(src, tgt) / 100.0
    return 0.6 * semantic_sim + 0.4 * fuzzy_sim


def schema_match_semantic(source_cols, target_cols, threshold=0.6):
    """Find best target column for each source column."""
    mapping = {}
    for src in source_cols:
        best_tgt, best_score = None, 0
        for tgt in target_cols:
            score = similarity_score(src, tgt)
            if score > best_score:
                best_score, best_tgt = score, tgt
        mapping[src] = best_tgt if best_score >= threshold else None
    return mapping

/Users/abd/Developer/team_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def unify_schemas(schema_dict, dataframes, target_schema_name="goodreads"):
    """
    Use one dataset (e.g., 'goodreads') as target schema.
    Match all other schemas to it, rename, and add missing columns.
    """
    if target_schema_name not in schema_dict:
        raise ValueError(f"Target schema '{target_schema_name}' not found in schema_dict")

    target_cols = schema_dict[target_schema_name]
    mappings = {}
    aligned_dfs = {}

    for name, cols in schema_dict.items():
        df = dataframes[name].copy()

        if name == target_schema_name:
            # Target schema maps to itself
            mappings[name] = {col: col for col in cols}
        else:
            mappings[name] = schema_match_semantic(cols, target_cols)
            rename_dict = {k: v for k, v in mappings[name].items() if v is not None}
            df = df.rename(columns=rename_dict)

        # Add missing columns as None
        for col in target_cols:
            if col not in df.columns:
                df[col] = None

        # Reorder columns to match target schema
        df = df[target_cols]
        aligned_dfs[name] = df

    return target_cols, mappings, aligned_dfs

In [5]:
# Schema dictionary
schemas = {
    "amazon": amazon_books.columns,
    "metabooks": metabooks.columns,
    "goodreads": goodreads.columns
}


dfs = {
    "amazon": amazon_books,
    "metabooks": metabooks,
    "goodreads": goodreads
}

# Align all schemas
target_schema, mappings, aligned_dfs = unify_schemas(schemas, dfs, target_schema_name="metabooks")

print("Target Schema:", target_schema)

print("\nSchema Mappings:")
for name, mapping in mappings.items():
    print(f"\n{name}:")
    for k, v in mapping.items():
        print(f"  {k:20} -> {v}")


print("\nAmazon after alignment:", aligned_dfs["amazon"].columns.tolist())
print("\nMetabooks after alignment:", aligned_dfs["metabooks"].columns.tolist())
print("\nGoodreads after alignment:", aligned_dfs["goodreads"].columns.tolist())

Target Schema: Index(['id', 'title', 'author_name', 'rating', 'num_rating', 'language',
       'genres', 'publisher', 'page_count', 'price', 'publish_year',
       'isbn_clean'],
      dtype='object')

Schema Mappings:

amazon:
  id                   -> id
  title                -> title
  book-author          -> author_name
  year-of-publication  -> publish_year
  publisher            -> publisher
  isbn_clean           -> isbn_clean

metabooks:
  id                   -> id
  title                -> title
  author_name          -> author_name
  rating               -> rating
  num_rating           -> num_rating
  language             -> language
  genres               -> genres
  publisher            -> publisher
  page_count           -> page_count
  price                -> price
  publish_year         -> publish_year
  isbn_clean           -> isbn_clean

goodreads:
  id                   -> id
  title                -> title
  author               -> author_name
  rating            

In [6]:
aligned_dfs["amazon"].to_parquet(DATA_DIR / "amazon_df.parquet", index=False)
aligned_dfs["metabooks"].to_parquet(DATA_DIR / "metabooks_df.parquet", index=False)
aligned_dfs["goodreads"].to_parquet(DATA_DIR / "goodreads_df.parquet", index=False)

In [7]:
amazon_books = pd.read_parquet(DATA_DIR / "amazon_df.parquet")
metabooks = pd.read_parquet(DATA_DIR / "metabooks_df.parquet")
goodreads = pd.read_parquet(DATA_DIR / "goodreads_df.parquet")

## Data Profiling

In [8]:
datasets = [amazon_books, metabooks, goodreads]
names = ["Amazon Books", "Metabooks", "Goodread Books"]

In [9]:
from PyDI.profiling import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df)

display(profile)

amazon_books:
  Rows: 271,360
  Columns: 12
  Total nulls: 1,628,165
  Null percentage: 50.0%
  Null counts per column:
    rating: 271,360 (100.0%)
    num_rating: 271,360 (100.0%)
    language: 271,360 (100.0%)
    genres: 271,360 (100.0%)
    publisher: 2 (0.0%)
    page_count: 271,360 (100.0%)
    price: 271,360 (100.0%)
    publish_year: 3 (0.0%)

metabooks:
  Rows: 535,159
  Columns: 12
  Total nulls: 453,739
  Null percentage: 7.1%
  Null counts per column:
    language: 3,501 (0.7%)
    genres: 33,841 (6.3%)
    publisher: 12,538 (2.3%)
    page_count: 52,142 (9.7%)
    price: 67,505 (12.6%)
    publish_year: 284,212 (53.1%)

goodreads:
  Rows: 52,478
  Columns: 12
  Total nulls: 18,565
  Null percentage: 2.9%
  Null counts per column:
    page_count: 2,511 (4.8%)
    price: 14,377 (27.4%)
    publish_year: 1,677 (3.2%)



{'rows': 52478,
 'columns': 12,
 'nulls_total': 18565,
 'nulls_per_column': {'id': 0,
  'title': 0,
  'author_name': 0,
  'rating': 0,
  'num_rating': 0,
  'language': 0,
  'genres': 0,
  'publisher': 0,
  'page_count': 2511,
  'price': 14377,
  'publish_year': 1677,
  'isbn_clean': 0},
 'dtypes': {'id': 'object',
  'title': 'object',
  'author_name': 'object',
  'rating': 'float64',
  'num_rating': 'Int64',
  'language': 'string',
  'genres': 'string',
  'publisher': 'string',
  'page_count': 'Int32',
  'price': 'float64',
  'publish_year': 'Int16',
  'isbn_clean': 'string'}}

## Entity Matching

In [10]:
train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [16]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators = [
    StringComparator(column='title',similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='title',similarity_function='levenshtein'),
    StringComparator(column='title',similarity_function='jaro_winkler'),
    
    StringComparator(column='author_name',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author_name',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author_name',similarity_function='levenshtein', preprocess=str.lower),

    StringComparator(column='publisher',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='levenshtein', preprocess=str.lower),
    
    StringComparator(column='isbn_clean',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='isbn_clean',similarity_function='levenshtein', preprocess=str.lower),

    NumericComparator(column='publish_year',max_difference=1),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower, 
                     list_strategy='concatenate'),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower,
                     list_strategy='best_match'),

    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2)
]

### Sorted Neighbourhood Blocker

In [17]:
from PyDI.entitymatching import SortedNeighbourhoodBlocker


blocker_m2a = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


blocker_m2g = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

sorted_candidates_m2a = blocker_m2a.materialize()
sorted_candidates_m2g = blocker_m2g.materialize()

### Evaluate Blocking

In [18]:
from PyDI.entitymatching import EntityMatchingEvaluator


# evaluate blocking: metabooks -> amazon
results_m2a = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=sorted_candidates_m2a,
    blocker=blocker_m2a,
    test_pairs=test_m2a,
    out_dir=BLOCK_EVAL_DIR
)


display(results_m2a)

{'pair_completeness': 0.9024390243902439,
 'pair_quality': 5.496907024126073e-06,
 'reduction_ratio': 0.9999536494738233,
 'total_candidates': 6731058,
 'total_possible_pairs': 145220746240,
 'true_positives_found': 37,
 'total_true_pairs': 41,
 'evaluation_timestamp': '2025-11-09T20:56:42.750656',
 'output_files': ['/Users/abd/Developer/team_project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/abd/Developer/team_project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

In [19]:
# evaluate blocking: metabooks -> goodreads
results_m2g = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=sorted_candidates_m2g,
    blocker=blocker_m2g,
    test_pairs=test_m2g,
    out_dir=BLOCK_EVAL_DIR
)


display(results_m2g)

{'pair_completeness': 0.8863636363636364,
 'pair_quality': 2.1702451597968205e-05,
 'reduction_ratio': 0.9999360124175761,
 'total_candidates': 1797032,
 'total_possible_pairs': 28084074002,
 'true_positives_found': 39,
 'total_true_pairs': 44,
 'evaluation_timestamp': '2025-11-09T20:58:37.676575',
 'output_files': ['/Users/abd/Developer/team_project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/abd/Developer/team_project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

## ML-based Matcher

In [20]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor = FeatureExtractor(comparators)

train_features_m2a = feature_extractor.create_features(
    metabooks, amazon_books, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor.create_features(
    metabooks, goodreads, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [22]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import pandas as pd

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


In [23]:
from PyDI.entitymatching import MLBasedMatcher


ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2a = ml_matcher.match(
    metabooks, amazon_books,
    candidates=blocker_m2a,
    id_column='id',
    trained_classifier=best_models[0]
)

correspondences_m2g = ml_matcher.match(
    metabooks, goodreads,
    candidates=blocker_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

In [25]:
correspondences_m2a.to_csv(CORR_DIR / "workflow_corr_m2a.csv", index=False)
correspondences_m2g.to_csv(CORR_DIR / "workflow_corr_m2g.csv", index=False)

### Evaluate Matching

In [26]:
debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

# evaluate matching metabooks -> amazon
eval_results_m2a = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2a,
    test_pairs=test_m2a,
    out_dir=debug_output_dir
)

display(eval_results_m2a)

{'precision': 1.0,
 'recall': 0.8536585365853658,
 'f1': 0.9210526315789475,
 'accuracy': 0.97,
 'true_positives': 35,
 'false_positives': 0,
 'false_negatives': 6,
 'true_negatives': 159,
 'threshold_used': 0.0,
 'total_correspondences': 25898,
 'filtered_correspondences': 25898,
 'evaluation_timestamp': '2025-11-09T22:02:46.349026',
 'output_files': ['/Users/abd/Developer/team_project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [27]:
# evaluate matching metabooks -> goodreads
eval_results_m2g = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2g,
    test_pairs=test_m2g,
    out_dir=debug_output_dir
)

display(eval_results_m2g)

{'precision': 0.918918918918919,
 'recall': 0.7727272727272727,
 'f1': 0.8395061728395061,
 'accuracy': 0.935,
 'true_positives': 34,
 'false_positives': 3,
 'false_negatives': 10,
 'true_negatives': 153,
 'threshold_used': 0.0,
 'total_correspondences': 5916,
 'filtered_correspondences': 5916,
 'evaluation_timestamp': '2025-11-09T22:02:53.700346',
 'output_files': ['/Users/abd/Developer/team_project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/abd/Developer/team_project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

### Cluster Analysis

In [29]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,25560,99.362463
1,3,157,0.610325
2,4,7,0.027212


In [30]:
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)


📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5783,98.871602
1,3,65,1.111301
2,4,1,0.017097


## Data Fusion

In [31]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

correspondences_m2a = pd.read_csv(CORR_DIR / "workflow_corr_m2a.csv")
correspondences_m2g = pd.read_csv(CORR_DIR / "workflow_corr_m2g.csv")

all_correspondences = pd.concat([correspondences_m2a, correspondences_m2g], ignore_index=True)

len(all_correspondences)

31814

In [34]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author_name', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [35]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_sn_blocker.jsonl")

fused_ml_snblocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_ml_snblocker):,}')
display(fused_ml_snblocker.head())

Fused rows: 30,185


,_id,_fusion_group_id,_fusion_sources,rating,isbn_clean,publisher,id,author_name,genres,num_rating,title,page_count,publish_year,price,language,_fusion_confidence,_fusion_metadata
0,metabooks_237547,group_0,"[metabooks, amazon_books]",4.7,0743400488,Atria,metabooks_237547,jim dutcher,"[['Science', 'Math', 'Biological Sciences']]",78,wolves at our door the extraordinary story of ...,320.0,2002.0,24.450001,English,0.818182,"{'rating_rule': 'prefer_higher_trust', 'rating..."
1,amazon_34710,group_1,"[metabooks, amazon_books]",4.2,0385491042,Anchor Books/Doubleday,amazon_34710,margaret atwood,"[['Literature', 'Fiction', 'History', 'Critici...",241,bluebeards egg stories,256.0,1998.0,15.470000,English,0.818182,"{'rating_rule': 'prefer_higher_trust', 'rating..."
2,metabooks_211703,group_2,"[metabooks, amazon_books]",3.4,0312960808,St Martins Pr (Mm),metabooks_211703,carlton smith,"[['Biographies', 'Memoirs', 'True Crime']]",19,blood money the du pont heir and the murder of...,269.0,1996.0,19.410000,English,0.818182,"{'rating_rule': 'prefer_higher_trust', 'rating..."
3,amazon_210620,group_3,"[metabooks, amazon_books]",5.0,0425097099,Berkley Publishing Group,amazon_210620,carol snyder,"[[""Children's Books""]]",1,the leftover kid,NaN,1987.0,5.990000,English,0.727273,"{'rating_rule': 'prefer_higher_trust', 'rating..."
4,metabooks_73697,group_4,"[metabooks, amazon_books]",3.6,0451455541,Roc,metabooks_73697,emily devenport,"[['Science Fiction', 'Fantasy']]",3,the kronos condition,336.0,1997.0,5.990000,English,0.818182,"{'rating_rule': 'prefer_higher_trust', 'rating..."


In [36]:
fused_ml_snblocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30185 entries, 0 to 30184
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 30185 non-null  object 
 1   _fusion_group_id    30185 non-null  object 
 2   _fusion_sources     30185 non-null  object 
 3   rating              30185 non-null  float64
 4   isbn_clean          30185 non-null  object 
 5   publisher           30185 non-null  object 
 6   id                  30185 non-null  object 
 7   author_name         30185 non-null  object 
 8   genres              28964 non-null  object 
 9   num_rating          30185 non-null  int64  
 10  title               30185 non-null  object 
 11  page_count          27473 non-null  float64
 12  publish_year        30171 non-null  float64
 13  price               27975 non-null  float64
 14  language            30069 non-null  object 
 15  _fusion_confidence  30185 non-null  float64
 16  _fus

In [37]:
fused_ml_snblocker.to_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")

### Fusion Evaluation

In [44]:
fused_dataset = pd.read_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= pd.read_parquet(MLDS_DIR / "golden_fused_books.parquet")

In [45]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd

def _is_missing_value(v):
    if v is None:
        return True
    if isinstance(v, float) and np.isnan(v):
        return True
    if isinstance(v, (list, tuple, set, np.ndarray)):
        return len(v) == 0
    if isinstance(v, str):
        s = v.strip().lower()
        return s in ("", "nan", "none")
    return False

def array_set_equality_match(fused_value, gold_value) -> bool:
    """Case-insensitive set equality for strings, arrays, or stringified lists."""
    if _is_missing_value(fused_value) and _is_missing_value(gold_value):
        return True
    if _is_missing_value(fused_value) or _is_missing_value(gold_value):
        return False

    def to_clean_set(value):
        # unwrap 0-d or 1-element numpy arrays
        if isinstance(value, np.ndarray):
            # flatten and collapse 1-element arrays
            flat = value.flatten().tolist()
            if len(flat) == 1:
                value = flat[0]
            else:
                value = flat

        # parse stringified lists like "['Fiction','Drama']"
        if isinstance(value, str):
            s = value.strip()
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set, np.ndarray)):
                    value = parsed
                else:
                    # simple delimited string
                    value = re.split(r"[|,;/]", s)
            except Exception:
                # not parsable, split manually
                value = re.split(r"[|,;/]", s)

        # flatten iterable containers
        if isinstance(value, np.ndarray):
            value = value.tolist()
        if isinstance(value, (list, tuple, set)):
            items = []
            for v in value:
                # recursively parse if element looks like "['a','b']"
                if isinstance(v, str) and v.strip().startswith("["):
                    try:
                        inner = ast.literal_eval(v)
                        if isinstance(inner, (list, tuple, set)):
                            items.extend(inner)
                            continue
                    except Exception:
                        pass
                items.append(v)
            return {str(x).strip().lower() for x in items if not _is_missing_value(x)}

        # scalar fallback
        return {str(value).strip().lower()}

    fused_set = to_clean_set(fused_value)
    gold_set  = to_clean_set(gold_value)

    if not fused_set and not gold_set:
        return True
    return fused_set == gold_set



strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author_name", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("rating", numeric_tolerance_match)
strategy.add_evaluation_function("numratings", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("genres", array_set_equality_match)





In [46]:
dupe_rows = fused_dataset[fused_dataset['isbn_clean'].duplicated(keep=False)]
dupe_rows.sort_values('isbn_clean')


,_id,_fusion_group_id,_fusion_sources,rating,isbn_clean,publisher,id,author_name,genres,num_rating,title,page_count,publish_year,price,language,_fusion_confidence,_fusion_metadata
1316,amazon_60391,group_1316,"[metabooks, amazon_books]",4.5,0307021459,Western Pub. Co,amazon_60391,annie north bedford,"[[""Children's Books""]]",26,walt disneys donald ducks toy sailboat a littl...,<NA>,1993,7.79,English,0.727273,"{'_id_inputs': [{'dataset': 'amazon_books', 'r..."
18141,metabooks_323815,group_18141,"[metabooks, goodreads]",5.0,0307021459,Golden Press,metabooks_323815,annie north bedford samuel armstrong adaptor,"[['Picture Books', 'Childrens', 'Animals', 'Fi...",1,donald ducks toy sailboat,24,1990,7.98,English,0.844008,"{'_id_inputs': [{'dataset': 'metabooks', 'reco..."
20884,metabooks_530857,group_20884,"[metabooks, goodreads]",4.7,0515087491,G.P. Putnam's Sons,metabooks_530857,w e b griffin,"[['Fiction', 'Historical Fiction', 'Military F...",2692,semper fi the corps book 1,352,1986,8.99,English,0.854895,"{'_id_inputs': [{'dataset': 'metabooks', 'reco..."
28128,amazon_79684,group_28128,"[metabooks, amazon_books]",4.6,0515087491,Jove Books,amazon_79684,w e b griffin,None,11,the corps semper fibk 1 corps paperback,<NA>,1988,6.57,English,0.659674,"{'_id_inputs': [{'dataset': 'amazon_books', 'r..."


In [47]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.610
  macro_accuracy: 0.613
  num_evaluated_records: 38828
  num_evaluated_attributes: 7
  total_evaluations: 264491
  total_correct: 161459
  rating_accuracy: 0.630
  rating_count: 38828
  publisher_accuracy: 0.341
  publisher_count: 38828
  genres_accuracy: 0.640
  genres_count: 37448
  title_accuracy: 0.499
  title_count: 38828
  page_count_accuracy: 0.678
  page_count_count: 35411
  publish_year_accuracy: 0.745
  publish_year_count: 38708
  price_accuracy: 0.756
  price_count: 36440

Overall Accuracy: 61.0%
